# GeoChat — Setup and Model Loading

[GeoChat](https://github.com/mbzuai-oryx/GeoChat) is the first open-source grounded large vision-language model (VLM) for remote sensing, from MBZUAI (CVPR 2024).

**Architecture:** LLaVA-1.5 (LLaMA-2-7B + CLIP-ViT-L visual encoder) fine-tuned on a 318k remote sensing instruction dataset covering:
- Visual question answering
- Scene classification and description
- Region-level grounding (spatial QA with bounding boxes)
- Multi-turn remote sensing dialogue

## Hardware Requirements

| Setup | VRAM needed | Load time |
|---|---|---|
| FP16 (full precision) | ~14 GB | ~60 s |
| INT8 (bitsandbytes 8-bit) | ~8 GB | ~90 s |
| INT4 (bitsandbytes 4-bit) | ~5 GB | ~120 s |
| CPU only | N/A | ~20+ min per query — not practical |

**Minimum recommended: GPU with 12 GB VRAM using INT8 quantization.**

## References
- GitHub: https://github.com/mbzuai-oryx/GeoChat
- Paper: https://arxiv.org/abs/2311.15826
- Model: https://huggingface.co/MBZUAI/GeoChat

## 1. Check GPU Availability

In [ ]:
import torch

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available:  {torch.cuda.is_available()}")

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"GPU:             {gpu_name}")
    print(f"VRAM:            {vram_gb:.1f} GB")

    if vram_gb >= 14:
        print("\nRecommended load mode: FP16")
        load_mode = "fp16"
    elif vram_gb >= 8:
        print("\nRecommended load mode: INT8 (bitsandbytes)")
        load_mode = "int8"
    else:
        print("\nRecommended load mode: INT4 (bitsandbytes) or use a larger GPU")
        load_mode = "int4"
else:
    print("\nNo GPU detected. GeoChat requires a GPU with ≥12 GB VRAM.")
    print("CPU-only inference is possible but takes 20+ minutes per query.")
    load_mode = "cpu"

## 2. Load HuggingFace Token

In [ ]:
import sys
import os

sys.path.insert(0, os.path.abspath("../../.."))
try:
    from dotenv_config import HF_TOKEN
    if HF_TOKEN:
        print("HF_TOKEN loaded from .env")
    else:
        print("HF_TOKEN not set. Request access at https://huggingface.co/MBZUAI/GeoChat")
except ImportError:
    HF_TOKEN = os.environ.get("HF_TOKEN", "")
    print(f"HF_TOKEN from environment: {'set' if HF_TOKEN else 'not set'}")

## 3. Load GeoChat Model

Model weights (~14 GB in FP16) are downloaded from HuggingFace on first use and cached in `~/.cache/huggingface/`.

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import torch

model_id = "MBZUAI/GeoChat"

print(f"Loading GeoChat in {load_mode} mode...")
print("(First run downloads ~14 GB — may take several minutes)")

tokenizer = AutoTokenizer.from_pretrained(
    model_id,
    use_fast=False,
    token=HF_TOKEN or None,
)

if load_mode == "fp16":
    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        torch_dtype=torch.float16,
        device_map="auto",
        token=HF_TOKEN or None,
    )
elif load_mode == "int8":
    bnb_config = BitsAndBytesConfig(load_in_8bit=True)
    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        quantization_config=bnb_config,
        device_map="auto",
        token=HF_TOKEN or None,
    )
elif load_mode == "int4":
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
    )
    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        quantization_config=bnb_config,
        device_map="auto",
        token=HF_TOKEN or None,
    )
else:  # CPU fallback
    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        torch_dtype=torch.float32,
        token=HF_TOKEN or None,
    )

model.eval()
print(f"\nModel loaded successfully.")
total_params = sum(p.numel() for p in model.parameters())
print(f"Parameters: {total_params/1e9:.2f}B")

## 4. Load the Visual Encoder (CLIP-ViT)

In [ ]:
from transformers import CLIPImageProcessor

image_processor = CLIPImageProcessor.from_pretrained(
    model_id,
    token=HF_TOKEN or None,
)
print(f"Image processor loaded.")
print(f"  Expected image size: {image_processor.size}")
print(f"  Crop size: {image_processor.crop_size}")

## 5. Smoke Test — Simple Inference

In [ ]:
import urllib.request
from PIL import Image

# Download a sample satellite image
test_url = "https://upload.wikimedia.org/wikipedia/commons/thumb/a/a7/Camponotus_flavomarginatus_ant.jpg/640px-Camponotus_flavomarginatus_ant.jpg"
test_image_path = "test_image.jpg"

if not os.path.exists(test_image_path):
    urllib.request.urlretrieve(test_url, test_image_path)

image = Image.open(test_image_path).convert("RGB")

# Process image
image_tensor = image_processor.preprocess(image, return_tensors="pt")["pixel_values"]
if torch.cuda.is_available():
    image_tensor = image_tensor.half().cuda()

# Format the GeoChat prompt (uses LLaVA-1.5 format)
prompt = "A chat between a curious user and an artificial intelligence assistant specializing in remote sensing. " \
         "USER: <image>\nDescribe what you see in this image in one sentence. ASSISTANT:"

inputs = tokenizer(prompt, return_tensors="pt")
if torch.cuda.is_available():
    inputs = {k: v.cuda() for k, v in inputs.items()}

with torch.no_grad():
    output = model.generate(
        **inputs,
        images=image_tensor,
        max_new_tokens=100,
        do_sample=False,
        temperature=1.0,
    )

response = tokenizer.decode(output[0], skip_special_tokens=True)
# Extract the assistant's response
if "ASSISTANT:" in response:
    response = response.split("ASSISTANT:")[-1].strip()

print(f"GeoChat response: {response}")

## 6. Optional: Flash Attention 2 for Faster Inference

Flash Attention 2 speeds up inference but requires a C++ compilation step (~10 minutes).

In [ ]:
print("To install Flash Attention 2 (optional, for faster inference):")
print("  uv run pip install flash-attn --no-build-isolation")
print("\nThen reload the model with:")
print('  model = AutoModelForCausalLM.from_pretrained(..., attn_implementation="flash_attention_2")')
print("\nExpect ~2x throughput improvement on long sequences.")